# DarkSight — final results, significance test and demo

Run this **after** notebooks 05, 06 and 07. It does three things:

1. **Collects every number** into the one table that goes in your report.
2. **Checks whether the difference is real** — a bootstrap test that answers "could this gap be luck?"
3. **Runs the live demo** you can show in the presentation, with a public link.

### Settings
**GPU T4 x2** · **Internet On** → **Run All** (~20 min).

### Required
Attach the outputs of notebooks 05 and 06 (**Add Input → Your Work → Notebook Output**) so that
`det_raw.pt` and `det_enh.pt` are available.

## Step 0 · Setup

In [ ]:
!pip -q install ultralytics gdown

import os, shutil, time, json
from pathlib import Path
from collections import Counter
import numpy as np, torch, cv2, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

START = time.time()
WORK, TEMP = Path('/kaggle/working'), Path('/kaggle/temp')
DEVICE = [0, 1]            # both T4s. Change to 0 if multi-GPU training misbehaves.
MAX_SIDE = 1280

CLASSES = ["Bicycle", "Boat", "Bottle", "Bus", "Car", "Cat",
           "Chair", "Cup", "Dog", "Motorbike", "People", "Table"]
CLASS_ID = {n.lower(): i for i, n in enumerate(CLASSES)}
print('GPUs:', torch.cuda.device_count())

In [ ]:
# Two small files from our repo: the official split list and the Zero-DCE weights.
!git clone -q --depth 1 https://github.com/Manideep-21/darksight.git {TEMP}/repo
SPLIT_FILE, ZERODCE_FILE = TEMP/'repo/configs/exdark_imageclasslist.txt', TEMP/'repo/weights/zerodce_epoch99.pth'

# ExDark itself (1.5 GB photos + 5 MB boxes) from the authors' Google Drive.
SRC = TEMP / 'exdark_src'
if not SRC.exists():
    !gdown -q 1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx -O {TEMP}/images.zip
    !gdown -q 1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i -O {TEMP}/boxes.zip
    !mkdir -p {SRC} && unzip -q -o {TEMP}/images.zip -d {SRC} && unzip -q -o {TEMP}/boxes.zip -d {SRC}
    !rm -rf {SRC}/__MACOSX

photos = {p.stem.lower(): p for p in (SRC/'ExDark').rglob('*') if p.is_file() and not p.name.startswith('.')}
boxes  = {p.name.lower().removesuffix('.txt').rsplit('.', 1)[0]: p for p in (SRC/'ExDark_Annno').rglob('*.txt')}

SPLIT_NAME = {'1': 'train', '2': 'val', '3': 'test'}
LIGHTING = {'1':'Low','2':'Ambient','3':'Object','4':'Single','5':'Weak',
            '6':'Strong','7':'Screen','8':'Window','9':'Shadow','10':'Twilight'}
split_of, lighting_of = {}, {}
for line in SPLIT_FILE.read_text().splitlines():
    c = line.split()
    if len(c) == 5 and c[1].isdigit():
        k = Path(c[0]).stem.lower()
        split_of[k], lighting_of[k] = SPLIT_NAME[c[4]], LIGHTING[c[2]]
print(len(photos), 'photos |', Counter(split_of.values()))

In [ ]:
def convert_one(photo_path, box_path, out_photo, out_label):
    """Save one photo as PNG and its boxes as a YOLO label file (centre-x, centre-y, w, h as fractions)."""
    img = Image.open(photo_path).convert('RGB')
    W, H = img.size
    lines = []
    for line in Path(box_path).read_text(errors='ignore').splitlines():
        parts = line.split()
        if not parts or parts[0].startswith('%'):
            continue
        name, l, t, w, h = parts[0], *map(float, parts[1:5])
        cls = CLASS_ID.get(name.lower())
        if cls is None:
            continue
        x1, y1 = max(l, 0), max(t, 0)
        x2, y2 = min(l + w, W), min(t + h, H)
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        lines.append(f"{cls} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
    if max(W, H) > MAX_SIDE:
        img.thumbnail((MAX_SIDE, MAX_SIDE))
    out_photo.parent.mkdir(parents=True, exist_ok=True)
    out_label.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_photo)
    out_label.write_text("\n".join(lines))

RAW = TEMP / 'data/raw'
for i, (key, photo) in enumerate(sorted(photos.items())):
    convert_one(photo, boxes[key], RAW/'images'/split_of[key]/f'{key}.png',
                RAW/'labels'/split_of[key]/f'{key}.txt')
    if i % 1500 == 0:
        print(f'{i}/{len(photos)}')
print('converted', len(photos), 'photos')

In [ ]:
import torch.nn as nn

class ZeroDCE(nn.Module):
    """Predicts 8 brightening curves; each is applied as  new = old + a*old*(1-old)."""
    def __init__(self, f=32):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.e_conv1 = nn.Conv2d(3, f, 3, 1, 1);     self.e_conv2 = nn.Conv2d(f, f, 3, 1, 1)
        self.e_conv3 = nn.Conv2d(f, f, 3, 1, 1);     self.e_conv4 = nn.Conv2d(f, f, 3, 1, 1)
        self.e_conv5 = nn.Conv2d(f*2, f, 3, 1, 1);   self.e_conv6 = nn.Conv2d(f*2, f, 3, 1, 1)
        self.e_conv7 = nn.Conv2d(f*2, 24, 3, 1, 1)
    def forward(self, x):
        x1 = self.relu(self.e_conv1(x));  x2 = self.relu(self.e_conv2(x1))
        x3 = self.relu(self.e_conv3(x2)); x4 = self.relu(self.e_conv4(x3))
        x5 = self.relu(self.e_conv5(torch.cat([x3, x4], 1)))
        x6 = self.relu(self.e_conv6(torch.cat([x2, x5], 1)))
        for r in torch.split(torch.tanh(self.e_conv7(torch.cat([x1, x6], 1))), 3, dim=1):
            x = x + r * (x * x - x)
        return x

zerodce = ZeroDCE().cuda().eval()
zerodce.load_state_dict(torch.load(ZERODCE_FILE, map_location='cuda'))

@torch.no_grad()
def brighten(img):
    x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().cuda() / 255.0
    return (zerodce(x).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255).astype('uint8')

In [ ]:
def write_yaml(path, train_dir, val_dir, test_dir):
    """Tell YOLO where the photos are. Absolute paths, so 'path:' is just the filesystem root."""
    names = "\n".join(f'  {i}: {n}' for i, n in enumerate(CLASSES))
    Path(path).write_text(f"path: /\ntrain: {train_dir}\nval: {val_dir}\ntest: {test_dir}\nnames:\n{names}\n")
    return str(path)

def draw_boxes(img, boxes_xyxy, labels):
    out = img.copy()
    for (x1, y1, x2, y2), text in zip(boxes_xyxy, labels):
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(out, text, (int(x1), max(int(y1)-5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return out

def find_in_inputs(filename):
    """Look for a file inside any attached Kaggle dataset / notebook output."""
    hits = sorted(Path('/kaggle/input').rglob(filename)) if Path('/kaggle/input').exists() else []
    return hits[0] if hits else None

In [ ]:
from ultralytics import YOLO
det_raw_path, det_enh_path = find_in_inputs('det_raw.pt'), find_in_inputs('det_enh.pt')
assert det_raw_path and det_enh_path, 'Attach the outputs of notebooks 05 and 06 first'
m_raw, m_enh = YOLO(det_raw_path), YOLO(det_enh_path)

ENH = TEMP/'data/enhanced'
shutil.rmtree(ENH/'labels/test', ignore_errors=True)
shutil.copytree(RAW/'labels/test', ENH/'labels/test')
(ENH/'images/test').mkdir(parents=True, exist_ok=True)
for i, p in enumerate(sorted((RAW/'images/test').glob('*.png'))):
    Image.fromarray(brighten(np.array(Image.open(p)))).save(ENH/'images/test'/p.name)
    if i % 800 == 0: print(f'{i}/2563')
print('test set brightened')

## Step 1 · The main table

In [ ]:
yaml_raw = write_yaml(TEMP/'raw.yaml', RAW/'images/train', RAW/'images/val', RAW/'images/test')
yaml_enh = write_yaml(TEMP/'enh.yaml', RAW/'images/train', RAW/'images/val', ENH/'images/test')

runs = {
    'E2  trained raw / tested raw':               (m_raw, yaml_raw),
    'E3  trained raw / tested brightened':        (m_raw, yaml_enh),
    'E4  trained brightened / tested brightened': (m_enh, yaml_enh),
}
final = {}
for name, (mdl, y) in runs.items():
    print('===', name)
    r = mdl.val(data=y, split='test', imgsz=640, device=0)
    final[name] = [r.box.map50, r.box.map, r.box.mp, r.box.mr]

final_table = pd.DataFrame(final, index=['mAP@0.5', 'mAP@0.5:0.95', 'precision', 'recall']).round(4)
print(final_table.to_string())
final_table.to_csv(WORK/'final_results.csv')

## Step 2 · Is the difference real, or luck?

We re-run the comparison on 1,000 random re-samples of the test set (drawing 2,563 photos with
repetition each time). If the gap keeps the same sign across nearly all of them, it is not noise.

To do that we need the detector's own boxes, and we score them with a short, readable implementation of
the standard rule: a box is correct if it has the right class and overlaps a true box by at least 50%.

In [ ]:
def load_true_boxes(labels_dir):
    """YOLO label files -> {photo: (classes, corner boxes as fractions)}."""
    truth = {}
    for p in sorted(Path(labels_dir).glob('*.txt')):
        cls, box = [], []
        for line in p.read_text().splitlines():
            c, xc, yc, w, h = line.split()
            xc, yc, w, h = map(float, (xc, yc, w, h))
            cls.append(int(c)); box.append([xc-w/2, yc-h/2, xc+w/2, yc+h/2])
        truth[p.stem] = (np.array(cls), np.array(box).reshape(-1, 4))
    return truth

def predict_all(model, images_dir):
    """Run the detector over a folder, keeping low-confidence boxes too (mAP needs them)."""
    out = {}
    files = sorted(Path(images_dir).glob('*.png'))
    for i in range(0, len(files), 32):
        chunk = files[i:i+32]
        for p, r in zip(chunk, model.predict([str(f) for f in chunk], conf=0.001, device=0, verbose=False)):
            h, w = r.orig_shape
            out[p.stem] = (r.boxes.cls.cpu().numpy().astype(int),
                           r.boxes.xyxy.cpu().numpy() / [w, h, w, h],
                           r.boxes.conf.cpu().numpy())
        if i % 640 == 0: print(f'{i}/{len(files)}')
    return out

truth = load_true_boxes(RAW/'labels/test')
pred_E2 = predict_all(m_raw, RAW/'images/test')
pred_E4 = predict_all(m_enh, ENH/'images/test')
print('predictions ready')

In [ ]:
def overlap(a, b):
    """IoU: shared area divided by combined area."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(2)
    area_a = (a[:, 2:] - a[:, :2]).prod(1)
    area_b = (b[:, 2:] - b[:, :2]).prod(1)
    return inter / (area_a[:, None] + area_b[None] - inter + 1e-12)

def match_photo(true_cls, true_box, pred_cls, pred_box, pred_conf):
    """Decide which predictions are correct. Most confident first; each true box can be used once."""
    order = np.argsort(-pred_conf)
    pred_cls, pred_box, pred_conf = pred_cls[order], pred_box[order], pred_conf[order]
    correct, used = np.zeros(len(pred_conf), bool), np.zeros(len(true_cls), bool)
    ious = overlap(pred_box, true_box)
    for i in range(len(pred_conf)):
        candidates = np.where((true_cls == pred_cls[i]) & ~used, ious[i], -1) if len(true_cls) else np.array([-1.0])
        j = int(candidates.argmax())
        if candidates[j] >= 0.5:
            correct[i], used[j] = True, True
    return pred_conf, pred_cls, correct, np.bincount(true_cls, minlength=12)

def mean_ap(matches):
    """Average precision at 50% overlap, averaged over the classes that appear."""
    conf = np.concatenate([m[0] for m in matches]); cls = np.concatenate([m[1] for m in matches])
    ok   = np.concatenate([m[2] for m in matches]); n_true = np.sum([m[3] for m in matches], 0)
    aps = []
    for c in range(12):
        if n_true[c] == 0:
            continue
        pick = cls == c
        if pick.sum() == 0:
            aps.append(0.0); continue
        o = ok[pick][np.argsort(-conf[pick])]
        hits, misses = np.cumsum(o), np.cumsum(~o)
        recall, precision = hits / n_true[c], hits / np.maximum(hits + misses, 1)
        r = np.concatenate(([0], recall, [recall[-1]], [1]))
        p = np.concatenate(([1], precision, [0], [0]))
        p = np.flip(np.maximum.accumulate(np.flip(p)))          # precision can only go down
        x = np.linspace(0, 1, 101)
        aps.append(np.trapezoid(np.interp(x, r, p), x))         # area under the curve
    return float(np.mean(aps))

keys = sorted(truth)
match_E2 = [match_photo(*truth[k], *pred_E2[k]) for k in keys]
match_E4 = [match_photo(*truth[k], *pred_E4[k]) for k in keys]
print(f'our mAP@0.5   E2 {mean_ap(match_E2):.4f}   E4 {mean_ap(match_E4):.4f}')
print('(should be within ~0.01 of the Ultralytics numbers in the table above)')

In [ ]:
rng = np.random.default_rng(0)
differences = []
for _ in range(1000):
    pick = rng.integers(0, len(keys), len(keys))              # same photos for both detectors
    differences.append(mean_ap([match_E4[i] for i in pick]) - mean_ap([match_E2[i] for i in pick]))
differences = np.array(differences)
low, high = np.percentile(differences, [2.5, 97.5])
observed = mean_ap(match_E4) - mean_ap(match_E2)

print(f'E4 - E2 = {observed:+.4f} mAP@0.5')
print(f'95% of re-samples fall between {low:+.4f} and {high:+.4f}')
print('The difference is REAL (the range excludes zero)' if low * high > 0 else
      'Cannot rule out luck (the range includes zero)')

plt.figure(figsize=(8, 3.5))
plt.hist(differences, bins=40, color='#3b6ea8')
plt.axvline(0, color='red', ls='--', label='no difference')
plt.xlabel('E4 - E2 mAP@0.5 across 1000 re-samples'); plt.legend(); plt.tight_layout()
plt.savefig(WORK/'bootstrap.png', dpi=150); plt.show()
pd.DataFrame({'observed': [observed], 'ci_low': [low], 'ci_high': [high]}).to_csv(WORK/'bootstrap.csv', index=False)

## Step 3 · Pictures for the report: where each pipeline wins

In [ ]:
gain = {k: int(match_E4[i][2].sum()) - int(match_E2[i][2].sum()) for i, k in enumerate(keys)}
best  = sorted(gain, key=gain.get, reverse=True)[:3]      # brightening helped most
worst = sorted(gain, key=gain.get)[:3]                    # brightening hurt most

def detect_and_draw(model, path, conf=0.25):
    img = np.array(Image.open(path))
    r = model.predict(img[:, :, ::-1], conf=conf, device=0, verbose=False)[0]
    labels = [f'{CLASSES[int(c)]} {s:.2f}' for c, s in zip(r.boxes.cls, r.boxes.conf)]
    return draw_boxes(img, r.boxes.xyxy.cpu().numpy(), labels)

for title, group in [('brightening HELPED', best), ('brightening HURT', worst)]:
    for k in group:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].imshow(detect_and_draw(m_raw, RAW/'images/test'/f'{k}.png'))
        axes[0].set_title(f'E2 raw — {k}'); axes[0].axis('off')
        axes[1].imshow(detect_and_draw(m_enh, ENH/'images/test'/f'{k}.png'))
        axes[1].set_title(f'E4 brightened ({title})'); axes[1].axis('off')
        plt.tight_layout(); plt.savefig(WORK/f'report_{title.split()[-1].lower()}_{k}.png', dpi=120); plt.show()

## Step 4 · How fast is it?
Speed matters for the "night-time dashcam" motivation in your report.

In [ ]:
sample = np.array(Image.open(RAW/'images/test'/f'{keys[0]}.png'))
def timed(fn, n=50):
    for _ in range(5): fn()
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return (time.perf_counter() - t) / n * 1000

t_bright = timed(lambda: brighten(sample))
t_detect = timed(lambda: m_enh.predict(sample[:, :, ::-1], device=0, verbose=False))
speed = pd.DataFrame({'milliseconds per photo': {'Zero-DCE brightening': round(t_bright, 1),
                                                 'YOLOv8 detection': round(t_detect, 1),
                                                 'both together': round(t_bright + t_detect, 1)}})
print(speed.to_string())
print(f'\nthat is {1000/(t_bright+t_detect):.1f} photos per second on one T4')
speed.to_csv(WORK/'speed.csv')

## Step 5 · Live demo

Creates a public link (valid ~72 h) you can open on any laptop or phone during the presentation.
Upload a dark photo and see both pipelines side by side.

In [ ]:
!pip -q install gradio
import gradio as gr

def run_demo(image, confidence):
    if image is None:
        return None, None, 'Upload a photo first.'
    raw = np.array(Image.fromarray(image).convert('RGB'))
    if max(raw.shape) > MAX_SIDE:
        im = Image.fromarray(raw); im.thumbnail((MAX_SIDE, MAX_SIDE)); raw = np.array(im)
    enhanced = brighten(raw)

    def detect(model, img):
        r = model.predict(img[:, :, ::-1], conf=confidence, device=0, verbose=False)[0]
        labels = [f'{CLASSES[int(c)]} {s:.2f}' for c, s in zip(r.boxes.cls, r.boxes.conf)]
        return draw_boxes(img, r.boxes.xyxy.cpu().numpy(), labels), len(labels)

    left, n_raw = detect(m_raw, raw)
    right, n_enh = detect(m_enh, enhanced)
    note = (f'Average brightness {brightness_of(raw):.2f} · '
            f'raw pipeline: {n_raw} objects · brightened pipeline: {n_enh} objects')
    return left, right, note

def brightness_of(img):
    rgb = img.astype(np.float32) / 255
    return float((0.299*rgb[...,0] + 0.587*rgb[...,1] + 0.114*rgb[...,2]).mean())

demo = gr.Interface(
    fn=run_demo,
    inputs=[gr.Image(label='Dark photo'), gr.Slider(0.05, 0.9, 0.25, label='Confidence')],
    outputs=[gr.Image(label='E2 — detection on the raw photo'),
             gr.Image(label='E4 — Zero-DCE, then detection'),
             gr.Textbox(label='Summary')],
    title='DarkSight — low-light object detection',
    description='Left: detector on the dark photo. Right: brightened with Zero-DCE first, then detected.',
    examples=[[str(RAW/'images/test'/f'{k}.png'), 0.25] for k in keys[:4]],
)
demo.launch(share=True)

In [ ]:
print('Files for your report (Output panel):')
for f in sorted(WORK.glob('*')):
    if f.is_file():
        print(f'  {f.name:34s} {f.stat().st_size/1e6:7.2f} MB')
print(f'\nTotal time: {(time.time()-START)/60:.0f} min')
print('''
Report checklist
  final_results.csv  -> the main table (E2 / E3 / E4)
  bootstrap.png/csv  -> proof the difference is not luck
  report_*.png       -> pictures where brightening helped and where it hurt
  speed.csv          -> milliseconds per photo
  E5_E6 files (notebook 07) -> why the result came out the way it did
''')